In [1]:
ls

activator_20260427_183622.log   kourier4.txt
activator_20260427_184825.log   kourier5.txt
activator_20260428_110244.log   kourier6.txt
activator_20260428_111639.log   kourier7.txt
activator_20260428_113449.log   kourier8.txt
activator_20260428_114707.log   kourier9.txt
activator_20260428_115946.log   kourier.csv
activator_20260428_121250.log   latency_percentiles1.csv
activator_20260428_122502.log   latency_percentiles2.csv
activator_20260428_123634.log   latency_percentiles3.csv
autoscaler_20260427_183622.log  latency_percentiles4.csv
autoscaler_20260427_184825.log  latency_percentiles5.csv
autoscaler_20260428_110244.log  latency_percentiles6.csv
autoscaler_20260428_111639.log  latency_percentiles7.csv
autoscaler_20260428_113449.log  latency_percentiles8.csv
autoscaler_20260428_114707.log  latency_percentiles9.csv
autoscaler_20260428_115946.log  latency_percentiles.csv
autoscaler_20260428_121250.log  logs/
autoscaler_20260428_122502.log  scheduler_20260427_183622.log
autoscaler_202604

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [3]:
files = [
    "invocation_results_fixed1.csv",
    "invocation_results_fixed2.csv",
    "invocation_results_fixed3.csv",
    "invocation_results_fixed4.csv",
    "invocation_results_fixed5.csv",
    "invocation_results_fixed6.csv",
    "invocation_results_fixed7.csv",
    "invocation_results_fixed8.csv",
    "invocation_results_fixed9.csv",
    "invocation_results_fixed10.csv"
]
kourier = [
   "kourier.csv"
]

pods = [
    "logs/containers_live_20260427_183611.csv",
    "logs/containers_live_20260427_184828.csv",
    "logs/containers_live_20260428_110241.csv",
    "logs/containers_live_20260428_111636.csv",
    "logs/containers_live_20260428_113446.csv",
    "logs/containers_live_20260428_114703.csv",
    "logs/containers_live_20260428_115939.csv",
    "logs/containers_live_20260428_121246.csv",
    "logs/containers_live_20260428_122458.csv",
    "logs/containers_live_20260428_123631.csv"
  
   
]



scheduler = pd.read_csv('scheduler_times.csv')

# concaténation

#invocation
dfs = [pd.read_csv(f) for f in files]
all_data = pd.concat(dfs, ignore_index=True)

#kourier
kourier_dfs = [pd.read_csv(f) for f in kourier]
kourier_df = pd.concat(kourier_dfs, ignore_index=True)

#pod queue-proxy
dfs = [pd.read_csv(f) for f in pods]  
pod_df = pd.concat(dfs, ignore_index=True)  

#autoscaler
#dfs = [pd.read_csv(f) for f in auto]  
#auto_df = pd.concat(dfs, ignore_index=True)  

#activator
#dfs = [pd.read_csv(f) for f in acti]  
#acti_df = pd.concat(dfs, ignore_index=True)  

#acti = pd.read_csv('context_handler_times1.csv')
#auto = pd.read_csv('reconcile_times1.csv')
#df_percentil = pd.read_csv('latency_percentiles.csv')


In [4]:
#conversion en ms

all_data['execution_time_sec'] = all_data['execution_time_sec'] * 1000


pod_df["createdAt"] = pd.to_datetime(pod_df["createdAt"], errors="coerce", utc=True)
pod_df["startedAt"] = pd.to_datetime(pod_df["startedAt"], errors="coerce", utc=True)
pod_df = pod_df.dropna(subset=["createdAt", "startedAt"])

# Vérification
print(pod_df.dtypes[["createdAt", "startedAt"]])

pod_df["duration_ms"] = (pod_df["startedAt"] - pod_df["createdAt"]).dt.total_seconds() * 1000
# Vérifier le résultat
print(pod_df[["pod", "container_name", "duration_ms"]].head())
pod_df.to_csv("pod_df_200ms_apiserver_pinned.csv", index=False)

createdAt    datetime64[ns, UTC]
startedAt    datetime64[ns, UTC]
dtype: object
                                                 pod container_name  \
0         graphgen-00001-deployment-545ff9c679-qzkpq    queue-proxy   
1         mobilenet-00001-deployment-5d6d9ff4b-2nxvc    queue-proxy   
2        grayscale-00001-deployment-6c9cd67ccc-vl2rg    queue-proxy   
3  fastfouriertransform-00001-deployment-6bbbdc9c...    queue-proxy   
4         pagerank-00001-deployment-54689999dc-hv5wb    queue-proxy   

   duration_ms  
0   110.901137  
1   123.207757  
2    92.289236  
3    99.153477  
4   120.521099  


In [5]:
all_data.to_csv("all_data_200ms_apiserver_pinned.csv", index=False)
#acti_df.to_csv("acti_df_10ms_1core.csv", index=False)
#auto_df.to_csv("auto_df_10ms_1core.csv", index=False)
kourier_df.to_csv("kourier_df_200ms_apiserver_pinned.csv", index=False)